# Rainfall Prediction with BiLSTM

This notebook presents a daily rainfall forecasting workflow using a Bidirectional Long Short-Term Memory (BiLSTM) model and multivariate weather observations. Experiment controls and visible analysis remain in the notebook; reusable implementation is provided by `train_bilstm_rainfall.py`.

Data source: the public run uses the included synthetic `sample_weather_data.csv`. The original private BMKG source, access conditions, and sample-data scope are documented in `README.md`.

Workflow summary:

| Item | Setting |
|---|---|
| Target / y | `RAINFALL 24H MM` |
| Input / X | Past weather variables plus past rainfall values |
| Time window | Default: 7-day window to predict the next day |
| Split strategy | Default: chronological 80% train and 20% test |
| Main metric | Chronological-holdout Normalized MAAPE (%), lower is better |
| Extra metrics | MAAPE, MAE, and RMSE |
| Model | Configurable stack of `Bidirectional(LSTM)` layers followed by `Dense(1)` |
| Data checks | Raw audit -> preprocessing -> cleaned data validation -> EDA |
| Compute runtime | CPU with configurable TensorFlow thread usage |

Step 1 is the experiment control panel. A top-to-bottom run reproduces the data checks, modeling comparison, selected result, predictions, and saved artifacts. Each sequence contains only past observations; future rainfall values are not used as inputs.


## 0. Notebook Setup

Imports the display libraries and reusable functions from `train_bilstm_rainfall.py`.


In [ ]:
%matplotlib inline

from argparse import Namespace
from datetime import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

from train_bilstm_rainfall import (
    DEFAULT_DATE_COL,
    DEFAULT_TARGET_COL,
    METRIC_DISPLAY_DECIMALS,
    NORMALIZED_DISPLAY_DECIMALS,
    SOURCE_DECIMAL_PLACES,
    validate_args,
    configure_tensorflow_cpu,
    format_tensorflow_cpu_report,
    load_and_prepare_dataframe,
    make_lagged_sequences,
    build_lagged_dataframe,
    chronological_split,
    scale_split_data,
    maape_angle_np,
    normalized_maape_percent_np,
    format_decimal,
    round_metric_columns,
    run_grid_search,
    cleanup_old_output_runs,
)


### Notebook Display Helpers

These notebook display helpers keep tables, short summaries, and plots consistent. They only change presentation; model training and metric calculations stay unchanged.


In [ ]:
def target_last_dataframe(frame, target_col=DEFAULT_TARGET_COL):
    ordered = frame.copy()
    if target_col in ordered.columns:
        ordered = ordered[[column for column in ordered.columns if column != target_col] + [target_col]]
    if target_col in ordered.index:
        ordered = ordered.loc[[index_value for index_value in ordered.index if index_value != target_col] + [target_col]]
    if "column" in ordered.columns and target_col in set(ordered["column"]):
        ordered = pd.concat([
            ordered[ordered["column"] != target_col],
            ordered[ordered["column"] == target_col],
        ], ignore_index=True)
    return ordered


def display_dataframe(frame, decimals=METRIC_DISPLAY_DECIMALS):
    displayed = frame.copy()
    numeric_cols = displayed.select_dtypes(include=["number"]).columns
    displayed[numeric_cols] = displayed[numeric_cols].round(decimals)
    return displayed


from IPython.display import display as _ipython_display
pd.set_option("display.float_format", lambda value: format_decimal(value))


def display(*objects, **kwargs):
    prepared = [
        display_dataframe(item) if isinstance(item, pd.DataFrame) else item
        for item in objects
    ]
    return _ipython_display(*prepared, **kwargs)


def source_precision_dataframe(frame):
    rounded = frame.copy()
    for column, decimals in SOURCE_DECIMAL_PLACES.items():
        if column in rounded.columns:
            rounded[column] = rounded[column].round(decimals)
    return target_last_dataframe(rounded)


def normalized_display_dataframe(frame):
    rounded = frame.copy()
    numeric_cols = rounded.select_dtypes(include=["number"]).columns
    rounded[numeric_cols] = rounded[numeric_cols].round(NORMALIZED_DISPLAY_DECIMALS)
    return target_last_dataframe(rounded)


def metric_display_dataframe(frame):
    return display_dataframe(round_metric_columns(frame, decimals=METRIC_DISPLAY_DECIMALS))


def display_summary(title, items, note=None):
    lines = [f"**{title}**", ""]
    for label, value in items:
        if isinstance(value, (float, np.floating)):
            value = format_decimal(value)
        elif isinstance(value, (list, tuple)):
            value = ", ".join(str(item) for item in value)
        lines.append(f"- **{label}:** {value}")
    if note:
        lines.extend(["", note])
    display(Markdown("\n".join(lines)))


def lagged_display_dataframe(frame):
    rounded = frame.copy()
    for source_column, decimals in SOURCE_DECIMAL_PLACES.items():
        matching_columns = [
            column for column in rounded.columns
            if column == source_column or column.endswith(f"_{source_column}")
        ]
        if matching_columns:
            rounded[matching_columns] = rounded[matching_columns].round(decimals)
    if "rainfall_target_mm" in rounded.columns:
        target_decimals = SOURCE_DECIMAL_PLACES.get(DEFAULT_TARGET_COL, METRIC_DISPLAY_DECIMALS)
        rounded["rainfall_target_mm"] = rounded["rainfall_target_mm"].round(target_decimals)
    return rounded

def apply_notebook_grid(ax, settings):
    if settings.grid:
        ax.grid(True)


def save_notebook_prediction_plot(dates, y_true, y_pred, output_path, title, settings, show_plot=False):
    fig, ax = plt.subplots(figsize=settings.figsize, dpi=settings.dpi)
    ax.plot(pd.to_datetime(dates), np.asarray(y_true).reshape(-1), label="Actual", linewidth=settings.line_width)
    ax.plot(pd.to_datetime(dates), np.asarray(y_pred).reshape(-1), label="Predicted", linewidth=settings.line_width)
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("24-hour rainfall (mm)")
    apply_notebook_grid(ax, settings)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=settings.dpi)
    if show_plot:
        plt.show()
    plt.close(fig)


## 1. Workflow Configuration

Edit this cell for ordinary experiments; the backend does not need to be changed.

| Group | Settings | Purpose |
|---|---|---|
| Data setup | `data`, `date_col`, `target_col` | Select the dataset and identify the target column. |
| Forecast design | `lag`, `train_ratio` | Set the look-back window and chronological split. |
| Grid search | `units`, `batch_sizes`, `lr_drop_periods` | Test the selected BiLSTM hyperparameter combinations. |
| Fixed training settings | `bilstm_layers`, `epochs`, `lr_drop_factor`, `initial_learning_rate`, `optimizer`, `loss_function` | Keep non-grid architecture and training choices consistent across runs. |
| Reproducibility and compute | `seed`, `cpu_threads` | Control repeatability and TensorFlow CPU parallelism. |

`units` is the Keras LSTM parameter applied to every stacked BiLSTM layer. `bilstm_layers` controls stack depth but is not part of the grid search. `lr_drop_periods` and `lr_drop_factor` are project-level scheduler settings implemented with Keras callbacks. Plot settings stay in their respective plotting cells.


In [ ]:
args = Namespace(
    # Select the dataset for this run.
    # Use Path("sample_weather_data.csv") for the included synthetic sample dataset.
    # Use Path("your_private_weather_data.csv") for an authorized private CSV locally.
    data=Path("sample_weather_data.csv"),
    date_col=DEFAULT_DATE_COL,
    target_col=DEFAULT_TARGET_COL,
    lag=7,
    train_ratio=0.80,
    units=[32, 64, 128],
    batch_sizes=[16, 32, 64],
    lr_drop_periods=[10, 20, 25],
    # General BiLSTM architecture/training settings. These are configurable but not part of the grid search.
    bilstm_layers=3,
    epochs=100,
    lr_drop_factor=0.1,
    # None keeps the selected Keras optimizer default.
    initial_learning_rate=None,
    optimizer="adam",
    loss_function="mse",
    seed=42,
    cpu_threads=-1,
    verbose=0,
    output_dir=Path("outputs"),
    keep_runs=1,
    show_plot=False,
    zero_codes=[8888.0, 9999.0],
    include_target_history=True,
    prepare_only=False,
)

validate_args(args)

output_dir = args.output_dir / f"notebook_bilstm_run_{datetime.now():%Y%m%d-%H%M%S}"

total_combinations = len(args.units) * len(args.batch_sizes) * len(args.lr_drop_periods)
display_summary(
    "Run plan",
    [
        ("Dataset", args.data.name),
        ("Input window", f"{args.lag} previous days"),
        ("BiLSTM layers", args.bilstm_layers),
        ("Hyperparameter combinations", total_combinations),
        ("Compute runtime", "CPU"),
        ("CPU threads", "all logical threads" if args.cpu_threads == -1 else args.cpu_threads),
        ("Saved runs kept", args.keep_runs),
    ],
    "This is the configuration that will be used when the training cell runs.",
)

## 2. Raw Data Audit

Raw-data inspection before preprocessing. This stage looks for issues that would affect modeling: missing values, duplicate dates, special codes (`8888` and `9999`), and rainfall target conditions.

Main plots are placed later in the workflow so they are not distorted by raw data-quality issues.

The issues found here are handled in Step 3, validated in Step 4, and analyzed in Step 5.


### 2.1 Raw Data Overview
Quick structural check of the raw dataset before any value is changed.


In [ ]:
raw_df = pd.read_csv(args.data)
raw_eda_df = raw_df.copy()
raw_eda_df[args.date_col] = pd.to_datetime(raw_eda_df[args.date_col], errors="coerce")

numeric_columns = [column for column in raw_eda_df.columns if column != args.date_col]
for column in numeric_columns:
    raw_eda_df[column] = pd.to_numeric(raw_eda_df[column], errors="coerce")

valid_dates = raw_eda_df[args.date_col].dropna()
display_summary(
    "Raw dataset overview",
    [
        ("Rows", len(raw_eda_df)),
        ("Columns", raw_eda_df.shape[1]),
        ("Date range", f"{valid_dates.min().date()} to {valid_dates.max().date()}" if len(valid_dates) else "Not available"),
        ("Duplicate dates", int(raw_eda_df.duplicated(subset=[args.date_col]).sum())),
        ("Numeric columns", len(numeric_columns)),
    ],
    "This quick check confirms the dataset size, coverage period, and basic structure before cleaning.",
)


### 2.2 Raw Dataset Table
Complete raw dataset table for direct inspection before preprocessing.


In [ ]:
display(Markdown("**Raw Dataset**"))
display(source_precision_dataframe(raw_eda_df))


### 2.3 Raw Data Issues to Fix
Missing values, special-code values, and rainfall target conditions handled in preprocessing.


In [ ]:
missing_eda_df = pd.DataFrame({
    "column": raw_eda_df.columns,
    "missing_values": raw_eda_df.isna().sum().values,
    "missing_percent": (raw_eda_df.isna().mean().values * 100).round(METRIC_DISPLAY_DECIMALS),
})
display(Markdown("**Missing Values Before Preprocessing**"))
display(missing_eda_df)

special_code_df = pd.DataFrame({
    "column": numeric_columns,
    "special_code_count": [int(raw_eda_df[column].isin(args.zero_codes).sum()) for column in numeric_columns],
})
display(Markdown("**Special Code Counts Before Preprocessing**"))
display(special_code_df)

rainfall_series = raw_eda_df[args.target_col]
rainfall_without_special_codes = rainfall_series.replace(args.zero_codes, np.nan)
rainfall_summary_df = pd.DataFrame({
    "item": [
        "missing_rainfall_days_raw",
        "special_code_rainfall_days_raw",
        "zero_rainfall_days_raw",
        "positive_valid_rainfall_days_raw",
        "maximum_valid_rainfall_mm_raw",
        "mean_valid_rainfall_mm_raw",
        "median_valid_rainfall_mm_raw",
    ],
    "value": [
        int(rainfall_series.isna().sum()),
        int(rainfall_series.isin(args.zero_codes).sum()),
        int((rainfall_series == 0).sum()),
        int((rainfall_without_special_codes > 0).sum()),
        round(float(rainfall_without_special_codes.max()), SOURCE_DECIMAL_PLACES[args.target_col]),
        round(float(rainfall_without_special_codes.mean()), METRIC_DISPLAY_DECIMALS),
        round(float(rainfall_without_special_codes.median()), SOURCE_DECIMAL_PLACES[args.target_col]),
    ],
})
display(Markdown("**Rainfall Summary Before Model Preprocessing**"))
display(rainfall_summary_df)


## 3. Data Preprocessing

Preprocessing fixes the raw-data issues found in Step 2 and builds the dataframe used by feature engineering and modeling.

| Raw-data issue | Handling rule |
|---|---|
| Special codes in X variables | `8888` and `9999` are converted to missing values and filled with linear interpolation |
| Missing X values | Filled with linear interpolation |
| Special codes or missing values in y/rainfall | Filled with `0` |
| Duplicate dates | Removed after chronological sorting |
| Date order | Sorted chronologically |
| Decimal precision | Rounded to the source-like decimal pattern defined in the training script |

The dataframe keeps X weather variables first and the rainfall target `RAINFALL 24H MM` last for readability. Historical rainfall is also included later as a lagged input feature, but only past rainfall values are used.

If an X column still cannot be filled by interpolation because it has no valid values, the workflow raises an error instead of filling X with `0`.


### 3.1 Apply Cleaning Rules
Applies the preprocessing rules and creates the dataframe used by the next stages.


In [ ]:
df, preprocessing_stats, feature_cols, exogenous_cols = load_and_prepare_dataframe(
    data_path=args.data,
    date_col=args.date_col,
    target_col=args.target_col,
    zero_codes=args.zero_codes,
    include_target_history=args.include_target_history,
)


display_summary(
    "Preprocessing complete",
    [
        ("Rows ready for modeling", len(df)),
        ("Target column", args.target_col),
        ("Input columns", len(feature_cols)),
    ],
    "The cleaned dataframe is ready for EDA and lag construction.",
)


### 3.2 Preprocessing Action Summary
Records what changed during preprocessing so the cleaning step remains transparent and reproducible.


In [ ]:
preprocessing_action_df = pd.DataFrame({
    "item": [
        "rows_after_chronological_sorting",
        "missing_daily_rows_inserted",
        "target_column_y",
        "original_x_columns",
        "model_input_columns",
        "special_codes_found_before_preprocessing",
        "x_special_codes_treated_as_missing",
        "y_special_codes_replaced_with_0",
        "remaining_x_missing_after_interpolation",
        "x_missing_fill_method",
        "y_missing_fill_method",
        "decimal_rounding_after_fill",
    ],
    "value": [
        len(df),
        preprocessing_stats["missing_daily_rows_inserted"],
        args.target_col,
        ", ".join(exogenous_cols),
        ", ".join(feature_cols),
        json.dumps(preprocessing_stats["zero_code_counts_before_preprocessing"]),
        json.dumps(preprocessing_stats["x_zero_codes_treated_as_missing"]),
        json.dumps(preprocessing_stats["y_zero_codes_replaced_with_zero"]),
        json.dumps(preprocessing_stats["remaining_x_empty_after_interpolation"]),
        preprocessing_stats["x_empty_fill_method"],
        preprocessing_stats["y_empty_fill_method"],
        json.dumps(preprocessing_stats["decimal_rounding_after_fill"]),
    ],
})

display(Markdown("**Preprocessing Actions Applied**"))
display(preprocessing_action_df)


### 3.3 Missing-Value Handling Check
Confirms whether the missing values found in the raw audit were handled correctly.


In [ ]:
missing_change_df = pd.DataFrame({
    "column": list(preprocessing_stats["empty_values_before_fill"].keys()),
    "missing_before_fill": list(preprocessing_stats["empty_values_before_fill"].values()),
    "missing_after_fill": list(preprocessing_stats["remaining_empty_values_after_fill"].values()),
})

display(Markdown("**Missing Values Before and After Fill**"))
display(missing_change_df)


## 4. Cleaned Data Validation

Quality-control stage for the dataframe produced by preprocessing. Missing values, special codes, duplicate dates, date gaps, negative rainfall values, and rainfall range are checked before EDA and feature construction.


### 4.1 Data Quality Check
Verifies that the cleaned dataset no longer contains the main issues found in the raw audit.


In [ ]:

expected_daily_rows = (
    (df.index.max() - df.index.min()).days + 1
    if len(df) and pd.notna(df.index.min()) and pd.notna(df.index.max())
    else 0
)
post_missing_df = pd.DataFrame({
    "column": df.columns,
    "missing_values": df.isna().sum().values,
    "missing_percent": (df.isna().mean().values * 100).round(METRIC_DISPLAY_DECIMALS),
})
post_special_code_df = pd.DataFrame({
    "column": df.columns,
    "special_code_count": [int(df[column].isin(args.zero_codes).sum()) for column in df.columns],
})
cleaned_rainfall = df[args.target_col]
post_check_df = pd.DataFrame({
    "item": [
        "rows",
        "start_date",
        "end_date",
        "duplicate_dates",
        "expected_daily_rows_between_start_and_end",
        "date_gap_count",
        "remaining_missing_values_total",
        "remaining_special_code_values_total",
        "negative_rainfall_values",
        "zero_rainfall_days",
        "positive_rainfall_days",
        "maximum_rainfall_mm",
        "mean_rainfall_mm",
        "median_rainfall_mm",
    ],
    "value": [
        len(df),
        df.index.min().date() if len(df) else None,
        df.index.max().date() if len(df) else None,
        int(df.index.duplicated().sum()),
        expected_daily_rows,
        int(max(expected_daily_rows - len(df), 0)),
        int(df.isna().sum().sum()),
        int(post_special_code_df["special_code_count"].sum()),
        int((cleaned_rainfall < 0).sum()),
        int((cleaned_rainfall == 0).sum()),
        int((cleaned_rainfall > 0).sum()),
        round(float(cleaned_rainfall.max()), SOURCE_DECIMAL_PLACES[args.target_col]),
        round(float(cleaned_rainfall.mean()), METRIC_DISPLAY_DECIMALS),
        round(float(cleaned_rainfall.median()), SOURCE_DECIMAL_PLACES[args.target_col]),
    ],
})

display(Markdown("**Data Quality Check**"))
display(post_check_df)

display(Markdown("**Missing Values**"))
display(post_missing_df)

display(Markdown("**Special Code Counts**"))
display(post_special_code_df)


### 4.2 Validated Modeling Dataset
Numeric summary and complete modeling table after the quality checks pass.


In [ ]:

display(Markdown("**Numeric Summary**"))
display(source_precision_dataframe(df.describe().T))

display(Markdown("**Modeling Dataset**"))
display(source_precision_dataframe(df.reset_index()))


## 5. Exploratory Data Analysis

EDA on the validated dataset. The focus is rainfall behavior, input-variable behavior, seasonal patterns, extreme events, and temporal signals before feature engineering and modeling.

These checks describe the dataset and motivate follow-up experiments, such as trying another lag value or reviewing difficult rainfall periods. They do not automatically determine the final lag, split, architecture, or hyperparameter grid; those choices are evaluated through model training and testing.


### 5.1 Rainfall Time Series
Rainfall variation over time, including dry periods, rainy periods, and high-rainfall spikes.


In [ ]:
plot_df = df.reset_index().sort_values(args.date_col).copy()
plot_cfg = Namespace(figsize=(15, 6), line_width=1.8, grid=True, dpi=180)

fig, ax = plt.subplots(figsize=plot_cfg.figsize, dpi=plot_cfg.dpi)
ax.plot(plot_df[args.date_col], plot_df[args.target_col], linewidth=plot_cfg.line_width)
ax.set_title("RAINFALL 24H MM")
ax.set_xlabel("Date")
ax.set_ylabel("Rainfall 24h (mm)")
apply_notebook_grid(ax, plot_cfg)
plt.tight_layout()
plt.show()


### 5.2 Input Variable Time Series
Time-series view of each X variable. The rainfall target is excluded here because it already has its own plot.


In [ ]:
x_variable_plot_df = df.reset_index().sort_values(args.date_col).copy()
x_variable_columns = [
    column for column in exogenous_cols
    if column in x_variable_plot_df.columns
]
plot_cfg = Namespace(width=18, row_height=3.8, line_width=1.4, grid=True, dpi=180)

subplot_cols = 2
subplot_rows = int(np.ceil(len(x_variable_columns) / subplot_cols))
fig, axes = plt.subplots(
    subplot_rows,
    subplot_cols,
    figsize=(plot_cfg.width, max(plot_cfg.row_height * subplot_rows, plot_cfg.row_height)),
    sharex=True,
    constrained_layout=True,
    dpi=plot_cfg.dpi,
)
axes = np.asarray(axes).reshape(-1)

for ax, column in zip(axes, x_variable_columns):
    ax.plot(
        x_variable_plot_df[args.date_col],
        x_variable_plot_df[column],
        linewidth=plot_cfg.line_width,
        color="#1f77b4",
    )
    ax.set_title(column)
    ax.set_xlabel("Date")
    ax.set_ylabel(column)
    apply_notebook_grid(ax, plot_cfg)
    ax.tick_params(axis="both")

for ax in axes[len(x_variable_columns):]:
    ax.set_visible(False)

plt.show()


### 5.3 Rainfall Distribution
Distribution of rainfall values, used to inspect how dominant zero and low-rainfall days are.


In [ ]:
plot_cfg = Namespace(figsize=(15, 6), bins=35, grid=True, dpi=180)

fig, ax = plt.subplots(figsize=plot_cfg.figsize, dpi=plot_cfg.dpi)
ax.hist(plot_df[args.target_col], bins=plot_cfg.bins, edgecolor="black", alpha=0.75)
ax.set_title("Rainfall Distribution")
ax.set_xlabel("Rainfall 24h (mm)")
ax.set_ylabel("Frequency")
apply_notebook_grid(ax, plot_cfg)
plt.tight_layout()
plt.show()


### 5.4 Feature Correlation Heatmap
Linear correlation overview between variables. This is useful for inspection, but it does not prove causal relationships.


In [ ]:

cleaned_correlation_df = df.corr(numeric_only=True)
display(Markdown("**Feature Correlation Matrix**"))
display(target_last_dataframe(cleaned_correlation_df.round(METRIC_DISPLAY_DECIMALS)))

plot_cfg = Namespace(min_width=9.5, col_width=1.2, min_height=6.5, row_height=0.85, dpi=180)

fig_width = max(plot_cfg.min_width, len(cleaned_correlation_df.columns) * plot_cfg.col_width)
fig_height = max(plot_cfg.min_height, len(cleaned_correlation_df.index) * plot_cfg.row_height)
fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=plot_cfg.dpi)
cleaned_correlation_plot_df = target_last_dataframe(cleaned_correlation_df)
image = ax.imshow(cleaned_correlation_plot_df, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_title("Feature Correlation Heatmap")
ax.set_xticks(np.arange(len(cleaned_correlation_plot_df.columns)))
ax.set_yticks(np.arange(len(cleaned_correlation_plot_df.index)))
ax.set_xticklabels(cleaned_correlation_plot_df.columns, rotation=45, ha="right")
ax.set_yticklabels(cleaned_correlation_plot_df.index)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


### 5.5 Calendar-Month Rainfall Distribution
Seasonal rainfall check using a monthly summary table and boxplot.


In [ ]:
display(Markdown("""**Calendar-Month Rainfall Distribution**

Calendar-month summary and boxplot are used to review seasonal rainfall variation across the available years. Wider boxes or higher outlying points indicate months with more variable or heavier rainfall, which can make those periods harder to model.
"""))

def _percentile(values, percentile):
    values = pd.Series(values).dropna()
    if len(values) == 0:
        return np.nan
    return float(np.percentile(values, percentile))

monthly_distribution_source_df = df[[args.target_col]].copy()
monthly_distribution_source_df["month_number"] = monthly_distribution_source_df.index.month
monthly_distribution_source_df["month_name"] = monthly_distribution_source_df.index.strftime("%b")
monthly_distribution_source_df["rainy_day"] = (monthly_distribution_source_df[args.target_col] > 0).astype(int)
monthly_distribution_df = (
    monthly_distribution_source_df
    .groupby(["month_number", "month_name"], as_index=False)
    .agg(
        days=(args.target_col, "size"),
        zero_rainfall_days=(args.target_col, lambda values: int((values == 0).sum())),
        rainy_days=("rainy_day", "sum"),
        rainfall_mean_mm=(args.target_col, "mean"),
        rainfall_median_mm=(args.target_col, "median"),
        rainfall_p95_mm=(args.target_col, lambda values: _percentile(values, 95)),
        rainfall_max_mm=(args.target_col, "max"),
    )
    .sort_values("month_number")
)
monthly_distribution_df["rainy_day_percent"] = monthly_distribution_df["rainy_days"] / monthly_distribution_df["days"] * 100
monthly_distribution_df = monthly_distribution_df[
    [
        "month_number",
        "month_name",
        "days",
        "zero_rainfall_days",
        "rainy_days",
        "rainy_day_percent",
        "rainfall_mean_mm",
        "rainfall_median_mm",
        "rainfall_p95_mm",
        "rainfall_max_mm",
    ]
]
for column in ["rainy_day_percent", "rainfall_mean_mm", "rainfall_p95_mm"]:
    monthly_distribution_df[column] = monthly_distribution_df[column].round(METRIC_DISPLAY_DECIMALS)
for column in ["rainfall_median_mm", "rainfall_max_mm"]:
    monthly_distribution_df[column] = monthly_distribution_df[column].round(SOURCE_DECIMAL_PLACES[args.target_col])
display(monthly_distribution_df)


boxplot_months = sorted(monthly_distribution_source_df["month_number"].unique())
boxplot_values = [
    monthly_distribution_source_df.loc[monthly_distribution_source_df["month_number"] == month, args.target_col]
    for month in boxplot_months
]
boxplot_labels = [pd.Timestamp(2024, int(month), 1).strftime("%b") for month in boxplot_months]
plot_cfg = Namespace(figsize=(12, 5), grid=True, dpi=180)

fig, ax = plt.subplots(figsize=plot_cfg.figsize, dpi=plot_cfg.dpi)
ax.boxplot(boxplot_values, tick_labels=boxplot_labels, showfliers=True)
ax.set_title("Rainfall Distribution by Calendar Month")
ax.set_xlabel("Calendar Month")
ax.set_ylabel("Rainfall 24h (mm)")
apply_notebook_grid(ax, plot_cfg)
plt.tight_layout()
plt.show()


### 5.6 Extreme Rainfall Check
High-rainfall behavior summarized with threshold counts, event dates, and a threshold plot.


In [ ]:
display(Markdown("""**Extreme Rainfall Check**

The upper tail of rainfall is summarized using thresholds calculated from positive-rainfall days only, so zero-rainfall days do not dilute heavy-rainfall behavior. Events at or above the positive-rainfall 95th percentile are listed because rare heavy-rainfall days often drive the largest forecasting errors.
"""))
positive_rainfall = cleaned_rainfall[cleaned_rainfall > 0]
extreme_rows = []


if len(positive_rainfall) > 0:
    for label, percentile in [("p90_positive_rainfall", 90), ("p95_positive_rainfall", 95), ("p99_positive_rainfall", 99)]:
        threshold = float(np.percentile(positive_rainfall, percentile))
        extreme_rows.append({
            "threshold": label,
            "rainfall_threshold_mm": round(threshold, SOURCE_DECIMAL_PLACES[args.target_col]),
            "days_at_or_above_threshold": int((cleaned_rainfall >= threshold).sum()),
            "percent_of_all_days": round(float((cleaned_rainfall >= threshold).mean() * 100), METRIC_DISPLAY_DECIMALS),
        })
    extreme_rows.append({
        "threshold": "maximum_rainfall",
        "rainfall_threshold_mm": round(float(cleaned_rainfall.max()), SOURCE_DECIMAL_PLACES[args.target_col]),
        "days_at_or_above_threshold": int((cleaned_rainfall == cleaned_rainfall.max()).sum()),
        "percent_of_all_days": round(float((cleaned_rainfall == cleaned_rainfall.max()).mean() * 100), METRIC_DISPLAY_DECIMALS),
    })
extreme_summary_df = pd.DataFrame(extreme_rows)
display(extreme_summary_df)

if len(positive_rainfall) > 0:
    p95_threshold = float(np.percentile(positive_rainfall, 95))
    extreme_event_df = (
        df.loc[df[args.target_col] >= p95_threshold, [args.target_col]]
        .reset_index()
        .rename(columns={args.date_col: "date", args.target_col: "rainfall_24h_mm"})
        .sort_values("rainfall_24h_mm", ascending=False)
    )
    extreme_event_df["rainfall_24h_mm"] = extreme_event_df["rainfall_24h_mm"].round(SOURCE_DECIMAL_PLACES[args.target_col])
    display(Markdown("**Extreme Rainfall Events at or Above Positive-Rainfall P95**"))
    display(extreme_event_df)

    plot_cfg = Namespace(figsize=(15, 6), line_width=1.6, threshold_line_width=2.0, grid=True, dpi=180)

    fig, ax = plt.subplots(figsize=plot_cfg.figsize, dpi=plot_cfg.dpi)
    ax.plot(plot_df[args.date_col], plot_df[args.target_col], linewidth=plot_cfg.line_width)
    ax.axhline(p95_threshold, color="red", linestyle="--", linewidth=plot_cfg.threshold_line_width, label="Positive rainfall P95")
    ax.set_title("Rainfall Time Series with Extreme-Rainfall Threshold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Rainfall 24h (mm)")
    ax.legend()
    apply_notebook_grid(ax, plot_cfg)
    plt.tight_layout()
    plt.show()


### 5.7 Rainfall Autocorrelation
Checks whether past rainfall values still have a relationship with future rainfall values at different lag distances.


In [ ]:
display(Markdown("""**Rainfall Autocorrelation / Lag Check**

Rainfall autocorrelation is calculated by lag to review whether past rainfall contains temporal signal for next-day prediction. Values near zero indicate weak linear dependence at that lag. This does not automatically select the final modeling lag, but it provides useful context for follow-up experiments.
"""))
max_lag_days = int(min(30, max(len(cleaned_rainfall) - 2, 1)))
autocorrelation_rows = []
for lag_day in range(1, max_lag_days + 1):
    autocorrelation_rows.append({
        "lag_days": lag_day,
        "rainfall_autocorrelation": cleaned_rainfall.autocorr(lag=lag_day),
        "selected_model_lag": lag_day == args.lag,
    })
autocorrelation_df = normalized_display_dataframe(pd.DataFrame(autocorrelation_rows))
display(autocorrelation_df)

plot_cfg = Namespace(figsize=(12, 5), bar_alpha=0.8, baseline_line_width=1.0, selected_lag_line_width=2.0, grid=True, dpi=180)

fig, ax = plt.subplots(figsize=plot_cfg.figsize, dpi=plot_cfg.dpi)
ax.bar(autocorrelation_df["lag_days"], autocorrelation_df["rainfall_autocorrelation"], alpha=plot_cfg.bar_alpha)
ax.axhline(0, color="black", linewidth=plot_cfg.baseline_line_width)
if args.lag <= max_lag_days:
    ax.axvline(args.lag, color="red", linestyle="--", linewidth=plot_cfg.selected_lag_line_width, label=f"Selected lag = {args.lag}")
    ax.legend()
ax.set_title("Rainfall Autocorrelation by Lag")
ax.set_xlabel("Lag (days)")
ax.set_ylabel("Autocorrelation")
apply_notebook_grid(ax, plot_cfg)
plt.tight_layout()
plt.show()


### 5.8 Lagged Feature-to-Target Correlation
Checks which lagged input features have the strongest direct linear relationship with the next-day rainfall target.


In [ ]:
display(Markdown("""**Lagged Feature-to-Target Correlation**

Lagged feature-to-target correlation provides a direct linear screening of past input values against the target rainfall day. For example, `lag_days = 1` compares yesterday's feature value with today's rainfall target. The result is treated as EDA context only; non-linear relationships may still be learned by the model.
"""))
lagged_feature_rows = []
for lag_day in range(1, args.lag + 1):
    for feature_name in feature_cols:
        lagged_feature_rows.append({
            "feature": feature_name,
            "lag_days": lag_day,
            "correlation_with_target_rainfall": df[feature_name].shift(lag_day).corr(df[args.target_col]),
        })
lagged_feature_correlation_df = normalized_display_dataframe(pd.DataFrame(lagged_feature_rows))
display(lagged_feature_correlation_df)

lagged_feature_pivot_df = lagged_feature_correlation_df.pivot(
    index="feature",
    columns="lag_days",
    values="correlation_with_target_rainfall",
)
lagged_feature_pivot_df = target_last_dataframe(lagged_feature_pivot_df)
display(Markdown("**Lagged Feature-to-Target Correlation Matrix**"))
display(lagged_feature_pivot_df)

plot_cfg = Namespace(min_width=8, col_width=1.25, min_height=4.8, row_height=0.6, dpi=180)

fig_width = max(plot_cfg.min_width, args.lag * plot_cfg.col_width)
fig_height = max(plot_cfg.min_height, len(lagged_feature_pivot_df.index) * plot_cfg.row_height)
fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=plot_cfg.dpi)
image = ax.imshow(lagged_feature_pivot_df, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_title("Lagged Feature-to-Target Correlation Heatmap")
ax.set_xlabel("Lag (days)")
ax.set_ylabel("Feature")
ax.set_xticks(np.arange(len(lagged_feature_pivot_df.columns)))
ax.set_xticklabels(lagged_feature_pivot_df.columns)
ax.set_yticks(np.arange(len(lagged_feature_pivot_df.index)))
ax.set_yticklabels(lagged_feature_pivot_df.index)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 6. Sliding Window Feature Construction

Converts the daily time series into supervised learning samples using the configured lag value.

For each sample, only past observations are used to predict rainfall on the next day. The target day itself is never included inside its input window.

Historical rainfall is part of the input features, but only as past rainfall. Next-day rainfall remains the prediction target. This keeps the forecasting setup chronological and avoids future information leakage.

The table below displays the full lagged dataset used for modeling. Lagged input columns are ordered from the oldest value in the window to the most recent value before the target day. The target date and `rainfall_target_mm` are placed at the far right so the input `X` columns and output `y` can be inspected directly.


In [ ]:
x_data, y_data, dates, sequence_stats = make_lagged_sequences(
    df=df,
    feature_cols=feature_cols,
    target_col=args.target_col,
    lag=args.lag,
)
lagged_df = build_lagged_dataframe(
    df=df,
    feature_cols=feature_cols,
    target_col=args.target_col,
    lag=args.lag,
)

display_summary(
    "Lagged dataset shape",
    [
        ("Input array X", str(x_data.shape)),
        ("Target array y", str(y_data.shape)),
        ("Skipped sequences with missing inputs", sequence_stats["skipped_nan_input"]),
    ],
    "Each X sample contains the configured number of previous days; the matching y value is the following day's rainfall.",
)

display(Markdown("**Complete lagged dataset used for modeling**"))
display(lagged_display_dataframe(lagged_df))

output_dir.mkdir(parents=True, exist_ok=True)
lagged_dataset_path = output_dir / "lagged_dataset.csv"
lagged_df.to_csv(lagged_dataset_path, index=False)


## 7. Train-Test Split, Normalization, and Denormalization

The default run uses a chronological 80% train and 20% test split. The ratio can be changed in the configuration cell, but the split remains chronological because this is time-series data.

Normalization flow:

1. Split the data into train and test sets chronologically.
2. Fit Min-Max normalization for `X` using train data only.
3. Apply the same train scaler to both train and test `X`.
4. Scale target `y` using the maximum absolute rainfall value from train data.
5. Train the BiLSTM using scaled `X` and scaled `y`.
6. Convert predictions back to millimeters before Normalized MAAPE (%), MAAPE, MAE, RMSE, CSV export, and plotting.

Input normalization formula:

```text
X_scaled = (X - X_train_min) / (X_train_max - X_train_min)
```

Target denormalization formula:

```text
y_pred_mm = y_pred_scaled * y_train_scale
```

The scaler is fitted only on train data to avoid test-set data leakage.

The code cell below displays the train-test split summary, the normalization values fitted from the training data, and the normalized train/test data. Each normalized row represents one target date. Columns named `t_minus_<n>` contain the configured input window, where `t_minus_1` is the day immediately before the target date.

The section also includes a **train-test rainfall distribution check** to compare rainfall patterns between the train and test periods. A wetter or more extreme test period can make the final test error harder to interpret.


### 7.1 Chronological Train-Test Split
Train and test periods, sample counts, and original/scaled array shapes.


In [ ]:
split_data = chronological_split(
    x_data=x_data,
    y_data=y_data,
    dates=dates,
    train_ratio=args.train_ratio,
)
scaled_data, scalers = scale_split_data(split_data)

split_summary = []
for split_name, (x_part, y_part, date_part) in split_data.items():
    x_scaled, y_scaled, _ = scaled_data[split_name]
    split_summary.append({
        "split": split_name,
        "samples": len(x_part),
        "start_date": pd.to_datetime(date_part[0]).date(),
        "end_date": pd.to_datetime(date_part[-1]).date(),
        "original_X_shape": x_part.shape,
        "scaled_X_shape": x_scaled.shape,
        "original_y_shape": y_part.shape,
        "scaled_y_shape": y_scaled.shape,
        "original_y_min_mm": float(np.min(y_part)),
        "original_y_max_mm": float(np.max(y_part)),
        "scaled_y_min": float(np.min(y_scaled)),
        "scaled_y_max": float(np.max(y_scaled)),
    })

display(normalized_display_dataframe(pd.DataFrame(split_summary)))


### 7.2 Normalization Parameters
Min-Max values fitted from training data only. Test data uses these training scalers.


In [ ]:
x_scaler_df = pd.DataFrame({
    "feature": feature_cols,
    "train_min_used_for_normalization": scalers["x_scaler"]["min"],
    "train_max_used_for_normalization": scalers["x_scaler"]["max"],
})
x_scaler_display_df = x_scaler_df.copy()
for row_index, feature_name in x_scaler_display_df["feature"].items():
    decimals = SOURCE_DECIMAL_PLACES.get(feature_name, NORMALIZED_DISPLAY_DECIMALS)
    x_scaler_display_df.loc[row_index, "train_min_used_for_normalization"] = round(
        x_scaler_display_df.loc[row_index, "train_min_used_for_normalization"], decimals
    )
    x_scaler_display_df.loc[row_index, "train_max_used_for_normalization"] = round(
        x_scaler_display_df.loc[row_index, "train_max_used_for_normalization"], decimals
    )

display(Markdown("**X normalization uses Min-Max values fitted from train data only.**"))
display(x_scaler_display_df)

display(Markdown(f"**Target y scale used during training:** `{format_decimal(scalers['y_scale'])} mm`."))
display(Markdown("Predictions are denormalized back to millimeters before Normalized MAAPE (%), MAAPE, MAE, RMSE, CSV export, and plotting."))


### 7.3 Normalized Train and Test Data
Complete normalized train and test sequences. Both tables use normalization parameters fitted from the training data only.


In [ ]:
def make_normalized_sequence_dataframe(x_scaled, y_scaled, date_part, feature_names):
    normalized_df = pd.DataFrame({
        "target_date": pd.to_datetime(date_part),
        "target_rainfall_scaled": y_scaled.reshape(-1),
    })
    lag_count = x_scaled.shape[1]
    for step_index in range(lag_count):
        lag_offset = lag_count - step_index
        for feature_index, feature_name in enumerate(feature_names):
            normalized_df[f"t_minus_{lag_offset}__{feature_name}"] = x_scaled[:, step_index, feature_index]
    return normalized_display_dataframe(normalized_df)

train_normalized_df = make_normalized_sequence_dataframe(
    scaled_data["train"][0],
    scaled_data["train"][1],
    scaled_data["train"][2],
    feature_cols,
)
test_normalized_df = make_normalized_sequence_dataframe(
    scaled_data["test"][0],
    scaled_data["test"][1],
    scaled_data["test"][2],
    feature_cols,
)

display(Markdown("**Normalized train data.**"))
display(train_normalized_df)

display(Markdown("**Normalized test data.**"))
display(test_normalized_df)


### 7.4 Train-Test Rainfall Distribution Check
Rainfall distribution comparison between train and test periods using summary statistics and distribution plots.


In [ ]:
display(Markdown("""**Train-Test Rainfall Distribution Check**

The target rainfall distribution is compared across the chronological train and test periods. Large differences in heavy-rainfall frequency or zero-rainfall percentage can explain why test performance differs from training performance.
"""))

def _split_target_summary(split_name, y_part, date_part):
    y_series = pd.Series(y_part.reshape(-1))
    positive = y_series[y_series > 0]
    return {
        "split": split_name,
        "samples": len(y_series),
        "start_date": pd.to_datetime(date_part[0]).date(),
        "end_date": pd.to_datetime(date_part[-1]).date(),
        "zero_rainfall_days": int((y_series == 0).sum()),
        "rainy_days": int((y_series > 0).sum()),
        "rainy_day_percent": round(float((y_series > 0).mean() * 100), METRIC_DISPLAY_DECIMALS),
        "rainfall_mean_mm": round(float(y_series.mean()), METRIC_DISPLAY_DECIMALS),
        "rainfall_median_mm": round(float(y_series.median()), SOURCE_DECIMAL_PLACES[args.target_col]),
        "rainfall_p95_mm": round(float(np.percentile(y_series, 95)), SOURCE_DECIMAL_PLACES[args.target_col]),
        "positive_rainfall_p95_mm": round(float(np.percentile(positive, 95)), SOURCE_DECIMAL_PLACES[args.target_col]) if len(positive) else 0.0,
        "rainfall_max_mm": round(float(y_series.max()), SOURCE_DECIMAL_PLACES[args.target_col]),
    }

train_test_distribution_df = pd.DataFrame([
    _split_target_summary("train", split_data["train"][1], split_data["train"][2]),
    _split_target_summary("test", split_data["test"][1], split_data["test"][2]),
])
display(train_test_distribution_df)

train_y = pd.Series(split_data["train"][1].reshape(-1), name="train")
test_y = pd.Series(split_data["test"][1].reshape(-1), name="test")
combined_max = max(float(train_y.max()), float(test_y.max()), 1.0)
bins = np.linspace(0, combined_max, 31)
plot_cfg = Namespace(figsize=(11, 5), grid=True, dpi=180)

fig, ax = plt.subplots(figsize=plot_cfg.figsize, dpi=plot_cfg.dpi)
ax.hist(train_y, bins=bins, alpha=0.6, label="Train", density=True)
ax.hist(test_y, bins=bins, alpha=0.6, label="Test", density=True)
ax.set_title("Train-Test Rainfall Distribution Comparison")
ax.set_xlabel("Rainfall 24h (mm)")
ax.set_ylabel("Density")
ax.legend()
apply_notebook_grid(ax, plot_cfg)
plt.tight_layout()
plt.show()

plot_cfg = Namespace(figsize=(8, 5), grid=True, dpi=180)

fig, ax = plt.subplots(figsize=plot_cfg.figsize, dpi=plot_cfg.dpi)
ax.boxplot([train_y, test_y], tick_labels=["Train", "Test"], showfliers=True)
ax.set_title("Train-Test Rainfall Boxplot")
ax.set_ylabel("Rainfall 24h (mm)")
apply_notebook_grid(ax, plot_cfg)
plt.tight_layout()
plt.show()


## 8. Normalized MAAPE (%) Metric

The main metric is **Normalized MAAPE (%)**, the normalized percentage form of **MAAPE (Mean Arctangent Absolute Percentage Error)**.

Formula:

```text
MAAPE = mean(arctan2(abs(y_true - y_pred), abs(y_true)))
Normalized MAAPE = MAAPE / (pi / 2)
Normalized MAAPE (%) = Normalized MAAPE * 100
```

`arctan2` is used to avoid manual division by actual rainfall. This keeps the metric defined when actual rainfall is `0`, which is important for rainfall datasets with many zero-rainfall days.

The metric is calculated after predictions are denormalized back to millimeters and negative rainfall predictions are clipped to `0`. MAAPE is also reported in its original angle scale, while MAE and RMSE are reported in millimeters as supporting metrics.


In [ ]:
maape_example = pd.DataFrame({
    "actual": [0.0, 0.0, 10.0],
    "predicted": [0.0, 5.0, 7.0],
})
maape_example["maape_angle"] = np.arctan2(
    np.abs(maape_example["actual"] - maape_example["predicted"]),
    np.abs(maape_example["actual"]),
)
maape_example["normalized_maape_percent_contribution"] = maape_example["maape_angle"] / (np.pi / 2) * 100

display(Markdown("**Row-level metric example**"))
display(normalized_display_dataframe(maape_example))
display_summary(
    "Example result",
    [
        ("MAAPE", maape_angle_np(maape_example["actual"], maape_example["predicted"])),
        ("Normalized MAAPE (%)", normalized_maape_percent_np(maape_example["actual"], maape_example["predicted"])),
    ],
    "The table shows each observation's contribution; the two values above summarize the complete example.",
)


## 9. BiLSTM Training and Hyperparameter Search

The grid search trains one BiLSTM model for every tested hyperparameter combination. Training uses normalized data, but final evaluation uses denormalized rainfall predictions in millimeters.

The best model is selected using the smallest Normalized MAAPE (%) on the chronological holdout.

The grid search focuses on three tuned hyperparameters: `units`, `batch_sizes`, and `lr_drop_periods`. Other training settings such as `epochs`, `lr_drop_factor`, `initial_learning_rate`, `optimizer`, and `loss_function` are configurable in the notebook, but they are applied consistently to every grid-search run rather than tested as grid dimensions.

`units` follows the Keras LSTM API name and is applied to each stacked BiLSTM layer. `bilstm_layers` is a fixed architecture setting for a run, while `lr_drop_periods` and `lr_drop_factor` are project-level learning-rate scheduler settings implemented with Keras callbacks, not hyperparameters of the LSTM layer itself. The LSTM and Bidirectional wrapper keep Keras defaults for settings such as activation, recurrent activation, dropout, recurrent dropout, initializers, and merge mode. Every BiLSTM layer except the final recurrent layer uses `return_sequences=True` so the next recurrent layer can receive a sequence input. The optimizer, loss function, epoch count, and layer count are taken from the workflow configuration and applied consistently to every tested combination.


### 9.1 Compute Runtime Report
Reports the CPU and TensorFlow thread configuration used for this run.


In [ ]:
tf, cpu_runtime = configure_tensorflow_cpu(
    args.seed, args.cpu_threads, print_report=False
)
display(Markdown(
    "**Compute runtime report**\n\n```text\n"
    + format_tensorflow_cpu_report(cpu_runtime)
    + "\n```"
))


### 9.2 Grid Search Results
This table lists every tested hyperparameter combination and the two MAAPE values used to compare them. Rows are sorted from the lowest Normalized MAAPE (%) to the highest.


In [ ]:
results_df, best = run_grid_search(
    args=args,
    scaled_data=scaled_data,
    original_data=split_data,
    scalers=scalers,
    output_dir=output_dir,
    tf=tf,
)

all_combinations_df = results_df[[
    "rank",
    "is_best",
    "bilstm_layers",
    "units",
    "batch_size",
    "lr_drop_period",
    "test_normalized_maape_percent",
    "test_maape",
]].rename(columns={
    "rank": "Rank",
    "is_best": "Best",
    "bilstm_layers": "BiLSTM Layers",
    "units": "Units",
    "batch_size": "Batch Size",
    "lr_drop_period": "LR Drop Period",
    "test_normalized_maape_percent": "Normalized MAAPE (%)",
    "test_maape": "MAAPE",
})
display(Markdown("**Chronological holdout metrics for all hyperparameter combinations**  \nEach row is one trained setup; the first row has the lowest Normalized MAAPE (%)."))
display(metric_display_dataframe(all_combinations_df))


### 9.3 Best Hyperparameter Combination
Compact holdout summary of the hyperparameter combination with the lowest Normalized MAAPE (%). Lower metric values are better.


In [ ]:
def compact_bilstm_setup(record):
    return (
        f"layers={int(record['bilstm_layers'])}, "
        f"units={int(record['units'])}, "
        f"batch_size={int(record['batch_size'])}, "
        f"lr_drop_period={int(record['lr_drop_period'])}"
    )

compact_best_df = pd.DataFrame([
    {
        "Model": "Best hyperparameter combination",
        "Normalized MAAPE (%)": best["test_normalized_maape_percent"],
        "MAAPE": best["test_maape"],
        "RMSE": best["test_rmse"],
        "MAE": best["test_mae"],
        "Setup": compact_bilstm_setup(best),
    }
])
compact_best_df.to_csv(output_dir / "best_model_summary.csv", index=False)

display(Markdown("**Best grid-search model summary**"))
display(metric_display_dataframe(compact_best_df))


## 10. Best Model Predictions
Predicted values are denormalized back to millimeters and saved as CSV. Negative rainfall predictions are clipped to `0` before evaluation, CSV export, and plotting.


In [ ]:
prediction_path = output_dir / "best_test_predictions.csv"

if prediction_path.exists():
    prediction_df = pd.read_csv(prediction_path)
    display(prediction_df)
else:
    display(Markdown("**Predictions are not available yet.** Run the training cell in Step 9 first."))


## 11. Actual vs Predicted Plot

Saved actual-vs-predicted rainfall plot from the selected hyperparameter combination. Values are shown in millimeters.


In [ ]:
plot_path = output_dir / "best_test_prediction_plot.png"
prediction_path = output_dir / "best_test_predictions.csv"

if prediction_path.exists():
    prediction_df = pd.read_csv(prediction_path)
    save_notebook_prediction_plot(
        dates=pd.to_datetime(prediction_df["date"]).to_numpy(),
        y_true=prediction_df["actual_rainfall_mm"].to_numpy(),
        y_pred=prediction_df["predicted_rainfall_mm"].to_numpy(),
        output_path=plot_path,
        title="Rainfall Prediction",
        settings=Namespace(figsize=(15, 6), line_width=2.0, grid=True, dpi=180),
        show_plot=False,
    )
    display(Image(filename=str(plot_path)))
elif plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    display(Markdown("**The prediction plot is not available yet.** Run the training cell in Step 9 first."))


## 12. Limitations

The detailed limitations and follow-up directions are documented in `README.md`. The constraints most relevant when interpreting this run are:

- the same chronological test period is used for hyperparameter selection and reporting, so it is not an untouched final holdout;
- linear interpolation is an offline cleaning rule and is not causal for real-time forecasting;
- one observation area, many zero-rainfall days, and sparse extreme events limit direct generalization.


## 13. Final Results and Output Files

The selected hyperparameter combination, holdout metrics, and run location are consolidated below. Combination-level details remain available in the modeling section and generated files.


In [ ]:
run_metadata = {
    "data_path": str(args.data),
    "date_col": args.date_col,
    "target_col": args.target_col,
    "lag": args.lag,
    "train_ratio": args.train_ratio,
    "feature_cols": feature_cols,
    "exogenous_cols": exogenous_cols,
    "include_target_history": args.include_target_history,
    "preprocessing_stats": preprocessing_stats,
    "sequence_stats": sequence_stats,
    "split_sizes": {name: int(len(values[0])) for name, values in split_data.items()},
    "scalers": scalers,
    "cpu_runtime": cpu_runtime,
    "selected_hyperparameters": {
        key: best[key]
        for key in ["bilstm_layers", "units", "batch_size", "lr_drop_period", "lr_drop_factor"]
    },
}
with (output_dir / "run_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2, default=str)

removed_dirs = cleanup_old_output_runs(args.output_dir, output_dir, args.keep_runs)

final_bilstm_results_df = compact_best_df.copy()
final_bilstm_results_df.insert(0, "Rank", 1)

display(Markdown("**Selected BiLSTM result**"))
display(metric_display_dataframe(final_bilstm_results_df))
display_summary(
    "Run summary",
    [
        ("Selected setup", compact_bilstm_setup(best)),
        ("Selection criterion", "Lowest chronological-holdout Normalized MAAPE (%)"),
        ("Input window", f"{args.lag} previous days"),
        ("Evaluation samples", f"{len(split_data['train'][0])} train / {len(split_data['test'][0])} test"),
        ("Output folder", str(output_dir)),
    ],
    "Detailed combination results, predictions, plots, and metadata are stored in the output folder.",
)
if removed_dirs:
    display(Markdown(f"Removed {len(removed_dirs)} older output run(s) according to `keep_runs`."))
